Hier werden nun die MAIAC-Daten der beiden MODIS-Sensoren Aqua und Terra mit einer 1 km Auflösung über GEE gedownloadet.


In [3]:
import ee
import os
import requests
import pandas as pd

# =========================================================
# 1. Earth Engine Initialisierung
# =========================================================
# Beim ersten Mal ausführen musst du die Auskommentierung der nächsten Zeile
# entfernen, um dich im Browser bei Google Earth Engine anzumelden:
ee.Authenticate()

print("Initialisiere Google Earth Engine...")
ee.Initialize()

# =========================================================
# 2. Einstellungen & Pfade
# =========================================================
OUTPUT_DIR = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\MAIAC_AOD"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = "2024-05-01"
END_DATE = "2024-08-31"

# Die Bounding Box von Würzburg (gleicher Extent wie bei ICON-D2)
LON_MIN, LON_MAX = 9.85, 10.05
LAT_MIN, LAT_MAX = 49.74, 49.85

# Geometrie für GEE erstellen
region = ee.Geometry.BBox(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)

# =========================================================
# 3. MAIAC AOD ImageCollection laden
# =========================================================
print("Lade MCD19A2.061 (MAIAC) Datensatz...")

# Wir nutzen das Band 'Optical_Depth_047' (Blaues Band),
# welches in der Forschung der Standard-Proxy für PM2.5 ist.
collection = (
    ee.ImageCollection("MODIS/061/MCD19A2_GRANULES")
    .filterBounds(region)
    .select('Optical_Depth_047')
)

# =========================================================
# 4. Tägliche Aggregation und Download
# =========================================================
# Wir iterieren durch jeden Tag, da wir eine tägliche Datei brauchen
days = pd.date_range(start=START_DATE, end=END_DATE, freq="D")
print(f"Starte Download-Prozess für {len(days)} Tage...\n")

for date in days:
    date_str = date.strftime("%Y-%m-%d")
    # GEE End-Date ist exklusiv, daher +1 Tag
    next_day_str = (date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    # Bilder für genau diesen Tag filtern
    daily_coll = collection.filterDate(date_str, next_day_str)

    # Prüfen, ob an diesem Tag Satellitendaten existieren (Wolken/Überflug)
    count = daily_coll.size().getInfo()

    if count == 0:
        print(f"[{date_str}] Keine Daten verfügbar (Evtl. dichte Wolkendecke). Überspringe...")
        continue

    try:
        # Tagesmittelwert bilden (Terra + Aqua Überflüge kombinieren)
        # WICHTIG: MAIAC AOD hat einen Skalierungsfaktor von 0.001!
        daily_mean = daily_coll.mean().multiply(0.001).clip(region)

        # Download-URL über die GEE-API anfordern
        # Wir fordern direkt EPSG:25832 an, um es kompatibel zum DOM zu machen!
        url = daily_mean.getDownloadURL({
            'scale': 1000,          # MAIAC hat 1km native Auflösung
            'crs': 'EPSG:25832',    # Transformation in dein metrisches System
            'region': region,
            'format': 'GEO_TIFF'
        })

        # Datei herunterladen und speichern
        filename = f"MAIAC_AOD_{date_str}.tif"
        filepath = os.path.join(OUTPUT_DIR, filename)

        response = requests.get(url)
        with open(filepath, 'wb') as f:
            f.write(response.content)

        print(f"[{date_str}] ERFOLG: {filename} heruntergeladen.")

    except Exception as e:
        print(f"[{date_str}] FEHLER beim Herunterladen: {e}")

print("\n" + "-" * 60)
print(f"DOWNLOAD ABGESCHLOSSEN! Alle Rasterdateien liegen unter:\n{OUTPUT_DIR}")
print("-" * 60)


Successfully saved authorization token.
Initialisiere Google Earth Engine...
Lade MCD19A2.061 (MAIAC) Datensatz...
Starte Download-Prozess für 123 Tage...

[2024-05-01] ERFOLG: MAIAC_AOD_2024-05-01.tif heruntergeladen.
[2024-05-02] ERFOLG: MAIAC_AOD_2024-05-02.tif heruntergeladen.
[2024-05-03] ERFOLG: MAIAC_AOD_2024-05-03.tif heruntergeladen.
[2024-05-04] ERFOLG: MAIAC_AOD_2024-05-04.tif heruntergeladen.
[2024-05-05] ERFOLG: MAIAC_AOD_2024-05-05.tif heruntergeladen.
[2024-05-06] ERFOLG: MAIAC_AOD_2024-05-06.tif heruntergeladen.
[2024-05-07] ERFOLG: MAIAC_AOD_2024-05-07.tif heruntergeladen.
[2024-05-08] ERFOLG: MAIAC_AOD_2024-05-08.tif heruntergeladen.
[2024-05-09] ERFOLG: MAIAC_AOD_2024-05-09.tif heruntergeladen.
[2024-05-10] ERFOLG: MAIAC_AOD_2024-05-10.tif heruntergeladen.
[2024-05-11] ERFOLG: MAIAC_AOD_2024-05-11.tif heruntergeladen.
[2024-05-12] ERFOLG: MAIAC_AOD_2024-05-12.tif heruntergeladen.
[2024-05-13] ERFOLG: MAIAC_AOD_2024-05-13.tif heruntergeladen.
[2024-05-14] ERFOLG: MAI